In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import  joblib

from src.evaluation import evaluate, seasonal_naive, select_best_model
from src.inventory import inventory_plan

print("Imports OK")

Imports OK


In [2]:
xgb_model = joblib.load('../models/xgboost_model.pkl')

df = pd.read_csv('../data/cleaned_data.csv', low_memory=False)
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfMonth'] = df['Date'].dt.day
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)

df['StoreType_encoded'] = df['StoreType'].map({'a': 0, 'b': 1, 'c': 2, 'd': 3})
df['Assortment_encoded'] = df['Assortment'].map({'a': 0, 'b': 1, 'c': 2})

features = ['Store', 'DayOfWeek', 'Year', 'Month', 'DayOfMonth', 'WeekOfYear',
            'Promo', 'SchoolHoliday', 'StoreType_encoded',
            'Assortment_encoded', 'CompetitionDistance', 'Promo2']

rows_before = len(df)
df = seasonal_naive(df)
print(f"Rows before baseline: {rows_before:,}  |  after: {len(df):,}")

test = df[df['Date'] >= '2015-04-01'].copy()
print(f"Test rows: {len(test):,}")
print(f"Test rows with no baseline (no sale 7 days earlier): "
      f"{test['Baseline_Prediction'].isna().sum():,}")

Rows before baseline: 1,017,155  |  after: 1,017,155
Test rows: 136,029
Test rows with no baseline (no sale 7 days earlier): 2


In [3]:
test = test.dropna(subset=['Baseline_Prediction']).copy()
test['XGB_Prediction'] = xgb_model.predict(test[features])

results = {
    'Seasonal Naive (baseline)': evaluate(test['Sales'], test['Baseline_Prediction']),
    'XGBoost': evaluate(test['Sales'], test['XGB_Prediction']),
}

results_df = pd.DataFrame(results).T.round(2)
print(f"Rows scored: {len(test):,}\n")
print(results_df)

mae_gain = (results['Seasonal Naive (baseline)']['MAE'] - results['XGBoost']['MAE']) \
           / results['Seasonal Naive (baseline)']['MAE'] * 100
print(f"\nXGBoost MAE vs baseline: {mae_gain:+.1f}%")

Rows scored: 136,027

                               MAE     RMSE   MAPE
Seasonal Naive (baseline)  2368.85  3541.40  34.59
XGBoost                    1580.64  2384.14  22.77

XGBoost MAE vs baseline: +33.3%


In [4]:
import xgboost as xgb
from src.features import FEATURES
from src.forecasting import time_split, tune_xgboost

train_all = df[df['Date'] < '2015-04-01']

fit_part, valid_part = time_split(train_all, '2015-01-01')
print(f"Tuning on: {len(fit_part):,} rows | Validation (Jan-Mar 2015): {len(valid_part):,} rows")

tuning_results = tune_xgboost(fit_part[FEATURES], fit_part['Sales'],
                              valid_part[FEATURES], valid_part['Sales'])
print(tuning_results)

Tuning on: 780,778 rows | Validation (Jan-Mar 2015): 100,348 rows
   max_depth  learning_rate  n_trees  valid_MAE  seconds
0          8           0.10      400     866.62       18
1          6           0.10      400     959.59       15
2          8           0.05      400     991.83       18
3          6           0.05      400    1145.00       15


In [5]:
from src.evaluation import mae

tuning_round2 = tune_xgboost(fit_part[FEATURES], fit_part['Sales'],
                             valid_part[FEATURES], valid_part['Sales'],
                             depths=(8, 10), learning_rates=(0.1, 0.2),
                             max_trees=1000)
print(tuning_round2)

default_model = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1,
                                 random_state=42, n_jobs=-1)
default_model.fit(fit_part[FEATURES], fit_part['Sales'])
default_valid_mae = mae(valid_part['Sales'], default_model.predict(valid_part[FEATURES]))
print(f"\nDefault settings (100 trees, depth 6, lr 0.1) on the same validation data: "
      f"MAE {default_valid_mae:.2f}")

   max_depth  learning_rate  n_trees  valid_MAE  seconds
0          8            0.2      422     795.83       22
1         10            0.1      644     846.54       40
2          8            0.1      567     851.03       27
3         10            0.2      254     919.10       17

Default settings (100 trees, depth 6, lr 0.1) on the same validation data: MAE 1338.88


In [6]:
best = tuning_round2.iloc[0]
print(best, "\n")

final_model = xgb.XGBRegressor(
    n_estimators=int(best['n_trees']),
    max_depth=int(best['max_depth']),
    learning_rate=float(best['learning_rate']),
    random_state=42, n_jobs=-1)
final_model.fit(train_all[FEATURES], train_all['Sales'])

test['XGB_Tuned_Prediction'] = final_model.predict(test[FEATURES])

results['XGBoost (tuned)'] = evaluate(test['Sales'], test['XGB_Tuned_Prediction'])
results_df = pd.DataFrame(results).T.round(2)
print(results_df)

THRESHOLD = 20
best_name, needs_tuning = select_best_model(results, THRESHOLD)
print(f"\nThreshold: MAPE below {THRESHOLD}%")
print(f"Selected model: {best_name}")
print(f"Did not qualify: {needs_tuning}")

max_depth          8.00
learning_rate      0.20
n_trees          422.00
valid_MAE        795.83
seconds           22.00
Name: 0, dtype: float64 

                               MAE     RMSE   MAPE
Seasonal Naive (baseline)  2368.85  3541.40  34.59
XGBoost                    1580.64  2384.14  22.77
XGBoost (tuned)            1121.78  1976.71  15.03

Threshold: MAPE below 20%
Selected model: XGBoost (tuned)
Did not qualify: ['Seasonal Naive (baseline)', 'XGBoost']


In [7]:
import joblib
joblib.dump(final_model, '../models/xgboost_tuned.pkl')
print("Saved")

Saved
